# Diffusion Models for Synthetic Fault-Code Images

**Use case:** Generate claim-photo-like machine display images (error codes `5E`, `UE`, …) when historical photos are sparse — same product story as the GAN lab, with **modern diffusion** methods.

**Package:** `ml/fault_code_diffusion/`
**Adjoins:**
- GAN lab: `fault_code_gan_synthetic_images.ipynb`
- Vision MLOps: `fault_code_vision_mlops_playbook.ipynb`
- RL: `fault_code_rl_playbook.ipynb`

---

## Why diffusion for *this* product?

| Need | How diffusion helps |
|------|---------------------|
| Sparse real claim photos | Offline **data factory** for OCR / vision eval |
| Class control (`5E` vs `UE`) | **Class-conditional** generation (label embedding) |
| Stable training | Denoising MSE — fewer GAN collapse issues |
| Client GPU story | Same CUDA path as OCR/DQN (`Dockerfile.ml`) |

**Still not on the diagnose hot path:** GraphRAG remains deterministic; diffusion only synthesizes training/test images (or future augmentation services).

## Part 1 — Theory (latest mainstream methods)

### 1.1 DDPM — Denoising Diffusion Probabilistic Models (Ho et al., NeurIPS 2020)

**Forward (destroy):** add Gaussian noise over \(T\) steps

\[
q(x_t \mid x_{t-1}) = \mathcal{N}\big(x_t; \sqrt{1-\beta_t}\, x_{t-1},\, \beta_t I\big)
\]

Closed form:

\[
x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon, \quad \varepsilon\sim\mathcal{N}(0,I)
\]

where \(\alpha_t = 1-\beta_t\), \(\bar\alpha_t = \prod_{s=1}^t \alpha_s\).

**Reverse (generate):** learn noise predictor \(\varepsilon_\theta(x_t, t, y)\) (optional class \(y\)):

\[
L = \mathbb{E}_{x_0,\varepsilon,t,y}\big\| \varepsilon - \varepsilon_\theta(x_t, t, y) \big\|_2^2
\]

### 1.2 Faster sampling

| Method | Idea |
|--------|------|
| **DDIM** (Song et al. 2021) | Non-Markovian reverse → 20–50 steps |
| **DPM-Solver / DPM-Solver++** | ODE solvers for diffusion → few steps |
| **Latent Diffusion (LDM)** (Rombach et al. CVPR 2022) | Diffuse in VAE latent space (Stable Diffusion family) |
| **Classifier-free guidance** (Ho & Salimans) | Mix conditional/unconditional scores for stronger control |

### 1.3 Product mapping

```text
y = fault code class (5E, UE, …)
x0 = LCD / claim-crop image
Train ε_θ on seeds + augments (+ real photos when available)
Sample many x0|y  →  OCR train / eval corpus  →  GraphRAG still uses extracted code
```

### 1.4 GAN vs diffusion (this monorepo)

See `ml.fault_code_diffusion.pipeline.diffusion_vs_gan()` in the next cells.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image
import torch
from torchvision import transforms
from torchvision.utils import make_grid

ROOT = Path(".").resolve()
if not (ROOT / "ml").is_dir() and (ROOT.parent / "ml").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))


from ml.fault_code_diffusion.device import device_report, pick_device
from ml.fault_code_diffusion.pipeline import diffusion_vs_gan, modern_stack_notes
from ml.fault_code_diffusion.ddpm import DDPMConfig, train_ddpm, generate_codes, p_sample_loop
from ml.fault_code_diffusion.schedule import DiffusionSchedule
from ml.fault_code_vision.pipeline import bootstrap_demo_dataset
from ml.fault_code_vision.catalog import FAULT_CATALOG, N_CLASSES, IDX_TO_CODE

ART = ROOT / "notebooks" / "fault_code_diffusion_artifacts"
ART.mkdir(parents=True, exist_ok=True)

print("device_report:", json.dumps(device_report(), indent=2))
print("device:", pick_device())
print("classes:", N_CLASSES, list(IDX_TO_CODE.values())[:8], "...")

import pandas as pd
print(pd.DataFrame(diffusion_vs_gan()).to_string(index=False))
print("\nUpgrade notes:")
print(json.dumps(modern_stack_notes(), indent=2))

## Part 2 — Forward process demo

Visualize a clean seed gradually noised to \(t \approx T\) (intuition for what the model learns to reverse).

In [ ]:
from ml.fault_code_vision.seed_render import render_lcd_panel
from torchvision.transforms import ToTensor, Normalize

img = render_lcd_panel("5E", style="lcd_dark", size=64)
x0 = Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))(ToTensor()(img)).unsqueeze(0)

device = pick_device()
sched = DiffusionSchedule(T=200, schedule="cosine", device=device)
x0 = x0.to(device)

ts = [0, 20, 50, 100, 150, 199]
fig, axes = plt.subplots(1, len(ts), figsize=(12, 2.2))
for ax, tval in zip(axes, ts):
    t = torch.tensor([tval], device=device)
    xt = sched.q_sample(x0, t)
    vis = (xt[0].detach().cpu() * 0.5 + 0.5).clamp(0, 1)
    ax.imshow(vis.permute(1, 2, 0).numpy())
    ax.set_title(f"t={tval}")
    ax.axis("off")
plt.suptitle("Forward diffusion on seed '5E'")
plt.tight_layout()
plt.savefig(ART / "forward_noise_5E.png", dpi=120)
plt.show()

## Part 3 — Train class-conditional DDPM (working code)

We train \(\varepsilon_\theta(x_t, t, y)\) on **full control-panel** seeds (readable digits).

**If samples look like pure noise:** the model is under-trained. Use ≥40–80 epochs (CUDA), or use the on-demand generator with `procedural` / `phone` methods for always-sharp images:

`notebooks/fault_code_image_generator.ipynb`

```bash
python -m ml.fault_code_vision.generate --code 5E --machine washer --methods procedural,phone --n 6
python -m ml.fault_code_diffusion.train --bootstrap --epochs 60 --require-cuda
```

In [ ]:
# Full control-panel seeds (readable digits). Under-trained DDPM ≈ pure noise.
# For crisp on-demand images without long train:
#   notebooks/fault_code_image_generator.ipynb  (methods procedural, phone)
from ml.fault_code_vision.cgan import build_panel_manifest

manifest = build_panel_manifest(ART / "panel_data", n_per_code=24, size=64)
print("panel manifest", manifest)

EPOCHS = 40  # 60–100 on CUDA for readable digits; short runs stay noisy
result = train_ddpm(
    manifest,
    ART,
    DDPMConfig(T=200, epochs=EPOCHS, batch_size=32, require_cuda=False),
)
print("checkpoint:", result["ckpt_path"])

hist = result["history"]
plt.figure(figsize=(7, 3))
plt.plot([h["epoch"] for h in hist], [h["loss"] for h in hist])
plt.xlabel("epoch")
plt.ylabel("noise MSE")
plt.title("DDPM training loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ART / "ddpm_loss.png", dpi=120)
plt.show()

## Part 4 — Sample synthetic claim-style codes

Generate multiple images per fault code for OCR augmentation.

In [ ]:
try:
    from IPython.display import display
except ImportError:
    def display(x):
        return None

codes = ["5E", "UE", "E24", "F9E1", "4E", "DE"]
paths = generate_codes(result["ckpt_path"], codes, ART / "generated", n_per_code=3)
print(f"wrote {len(paths)} images ->", ART / "generated")

fig, axes = plt.subplots(len(codes), 3, figsize=(6, 10))
for i, code in enumerate(codes):
    for j in range(3):
        p = ART / "generated" / f"diff__{code}__{j:03d}.png"
        if p.exists():
            axes[i, j].imshow(Image.open(p))
        axes[i, j].axis("off")
        if j == 0:
            axes[i, j].set_ylabel(code)
plt.suptitle("Conditional DDPM samples (quality rises with more epochs / CUDA)")
plt.tight_layout()
plt.savefig(ART / "preview_diffusion_grid.png", dpi=120)
plt.show()

samples = sorted((ART / "samples").glob("epoch_*.png"))
if samples:
    print("training snapshot:", samples[-1])
    display(Image.open(samples[-1]))
else:
    print("No training sample grids found yet.")

## Part 5 — Production / “latest methods” upgrade path

This notebook trains a **small from-scratch DDPM** so the client sees the math run end-to-end without multi‑GB downloads.

For **photo-real** laundry-room claim images, the usual 2024–2026 stack is:

1. **Latent diffusion** (Stable Diffusion family) via [Hugging Face Diffusers](https://github.com/huggingface/diffusers)
2. **ControlNet** / IP-Adapter — lock layout to LCD panel geometry
3. **LoRA fine-tune** on real claim crops when they arrive
4. **DDIM / DPM-Solver++** — fast sampling
5. **Classifier-free guidance** — stronger adherence to code text

Optional install (not required for this notebook’s DDPM):

```bash
pip install diffusers transformers accelerate
```

### Pipeline in the product (unchanged)

```text
Diffusion (offline) → synthetic labelled images
        → OCR train/eval (vision playbook)
        → extract code → GraphRAG Cypher / INDICATES boost
```

### Hardware

| Stage | Recommendation |
|-------|----------------|
| Showcase DDPM (this nb) | MPS/CPU OK for short train; CUDA better |
| SD / LDM fine-tune | **NVIDIA CUDA**, 12–24GB+ VRAM typical |
| Shared image | `docker/Dockerfile.ml` |

### References

- Ho et al. — *Denoising Diffusion Probabilistic Models* (2020)
- Song et al. — DDIM (2021)
- Rombach et al. — *High-Resolution Image Synthesis with Latent Diffusion Models* (2022)
- Nichol & Dhariwal — improved DDPM / cosine schedule
- Diffusers docs — practical production sampling

In [ ]:
checklist = [
    "Keep diffusion offline as data factory (not diagnose reasoner)",
    "Condition on closed-set fault-code labels from catalog",
    "Tag source=diffusion_synthetic in manifests",
    "Gate OCR on real photos before production pin",
    "Client train: CUDA + longer epochs / larger UNet or Diffusers LDM",
    "Optional: phone-camera augments after sampling (same as GAN lab)",
]
for i, c in enumerate(checklist, 1):
    print(f"{i}. {c}")

summary = {
    "notebook": "fault_code_diffusion_playbook.ipynb",
    "package": "ml/fault_code_diffusion",
    "method": "class_conditional_DDPM",
    "upgrade": "latent_diffusion_diffusers_controlnet",
    "cuda": "recommended_for_client_quality",
}
(ART / "executive_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))